In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

project_name = "wanted_2nd_agent_study"
os.environ["LANGSMITH_PROJECT"] = project_name

In [2]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from langchain_tavily import TavilySearch

### 빌트인 도구
- 랭체인 제공
- 도구와 툴킷
- 툴 -> 단일도구, 툴킷 -> 여러 도구가 묶여있다.
- 파일 관리 툴킷: 파일 삭제 툴, 파일 읽기 툴 ...

## 1. 검색 툴 사용해보기

In [26]:
TAVILY_PROMPT = """

너는 웹 검색 툴만 사용해서 사실을 확인하고 답하는 어시스턴트야.

[규칙]
1. 최신 뉴스/링크는 Tavily로 검색한다.
2. 모든 응답은 한국어로 한다.

"""

In [ ]:
# 1. 모델 설정
llm = ChatOpenAI(
    model= "gpt-4.1-mini",
    temperature=0
)

# 2. 도구 설정
tavily_search = TavilySearch(max_result=5)

# 3. 프롬프트 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", TAVILY_PROMPT),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

# 4. 단일 에이전트 생성
agent = create_openai_tools_agent(
    llm=llm,
    tools = [tavily_search],
    prompt = prompt
)

# 5. excutor 설정
excutor = AgentExecutor(
    agent = agent,
    tools = [tavily_search],
    verbose = True
)

In [36]:
result = excutor.invoke({"input": "애플 사의 최근 소식을 알려줘"})



> Entering new AgentExecutor chain...

Invoking: `tavily_search` with `{'query': '애플 최근 소식', 'time_range': 'week', 'topic': 'general'}`


{'query': '애플 최근 소식', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://kr.investing.com/news/company-news/article-93CH-1670871', 'title': '애플, M5 칩 공개…AI 성능 4배 향상 - 인베스팅닷컴', 'content': '다른 최근 소식으로 애플은 새로운 M5 칩을 탑재한 업그레이드된 Vision Pro 헤드셋을 10월 22일에 출시한다고 발표했다. 이 장치는 향상된 성능과 배터리 수명을 약속하며', 'score': 0.8156003, 'raw_content': None}, {'url': 'https://www.apple-economy.com/news/articleList.html?sc_day=7&sc_order_by=C&view_type=sm', 'title': '전체 < 기사목록 - 애플경제', 'content': "기사목록 보기 형태 ; 金대비 BTC 재평가, BTC \u200b\u200b16만 5천 달러까지 · 블록체인 엄정원 기자 2025.10.04 13:17 ; (2-②) 美캘리포니아 'AI안전법'-“'뉴욕주법' 비해 규제 약화”.", 'score': 0.559411, 'raw_content': None}, {'url': 'https://news.nate.com/view/20251015n00411', 'title': '"애플 이제는 한계" 전 CEO가 뼈 때렸다…증시불장 소외된이유 있었네', 'content': '팀 쿡 애플 CEO는 최근 실적 발표에서 “AI 경쟁에서 뒤처지지 않기 위해 인력 재편과 인수 확대를 추진하고 있다

## 2. 파일 저장하는 툴 사용해보기

In [3]:
# 1. 파일 저장 툴 가져오기
from langchain_community.agent_toolkits import FileManagementToolkit

root_dir = "../10_agent_result"
file_tools = FileManagementToolkit(
    root_dir=root_dir
).get_tools() # get_tools() 해야지 파일 매니저 툴을 가져올 수 있다.


file_tools


[CopyFileTool(root_dir='../10_agent_result'),
 DeleteFileTool(root_dir='../10_agent_result'),
 FileSearchTool(root_dir='../10_agent_result'),
 MoveFileTool(root_dir='../10_agent_result'),
 ReadFileTool(root_dir='../10_agent_result'),
 WriteFileTool(root_dir='../10_agent_result'),
 ListDirectoryTool(root_dir='../10_agent_result')]

In [3]:
# 2. 웹 서치 툴 가져오기
tavily_search = TavilySearch(max_result=5)

In [17]:
total_tool = file_tools + [tavily_search]
total_tool

[CopyFileTool(root_dir='../10_agent_result'),
 DeleteFileTool(root_dir='../10_agent_result'),
 FileSearchTool(root_dir='../10_agent_result'),
 MoveFileTool(root_dir='../10_agent_result'),
 ReadFileTool(root_dir='../10_agent_result'),
 WriteFileTool(root_dir='../10_agent_result'),
 ListDirectoryTool(root_dir='../10_agent_result'),
 TavilySearch(api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None))]

In [22]:
# 시스템 프롬프트 설정하기
SYSTEM_PROMPT = """
너는 '파일 시스템 툴'과 '웹 검색 툴' 만을 사용하는 어시스턴트야.

[규칙]
1. 최신 뉴스/링크는 Tavily로 검색한다.
2. 사용자가 csv 파일로 저장을 요청하면, 다음 필드들을 헤더로 csv를 만든다.
    date, query, title, url, source, snippet
3. 파일 명은 {YYYYMMDD}.csv 로 저장한다.
4. 같은 파일명이 있다면, 덮어 쓰지 말고, 기존 내용을 읽어서 헤더를 유지한 채 받아온 데이터를 모두 이어 붙여서 저장한다.
5.저장한 뒤에 내용을 확인해서 가장 아래 5줄 정보를 출력한다.

모든 답변은 한국어로 간결하게 한다.
"""

In [8]:
import datetime

def get_today():
    return datetime.datetime.now().strftime("%Y%m%d")

In [9]:
get_today()

'20251016'

In [5]:
# 1. 모델 설정
llm = ChatOpenAI(
    model = "gpt-4.1-mini",
    temperature = 0
)

# 2. 도구 생성
# total_tools

# 3. 프롬프트 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

# 4. 단일 에이전트 생성
agent = create_openai_tools_agent(
    llm=llm,
    tools = total_tool,
    prompt = prompt
)

# 5. excutor 설정
executor = AgentExecutor(
    agent = agent,
    tools = total_tool,
    verbose = True
)

NameError: name 'SYSTEM_PROMPT' is not defined

In [19]:
result = executor.invoke({
    "input": "오늘 AI 관련 뉴스 최신으로 5개 요약해줘.",
    "YYYYMMDD": get_today()
})



> Entering new AgentExecutor chain...

Invoking: `tavily_search` with `{'query': 'AI', 'search_depth': 'advanced', 'time_range': 'day', 'topic': 'general'}`


{'query': 'AI', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://medium.com/@mvineetsharma/artificial-intelligence-from-basic-concepts-to-agentic-systems-8ae3e8438bdd', 'title': 'Artificial Intelligence: From Basic Concepts to Agentic Systems', 'content': 'Artificial Intelligence (AI) represents humanity’s quest to create machines that can perform tasks typically requiring human intelligence. The concept dates back to ancient times, but the modern field emerged in the 1950s with pioneers like Alan Turing, who asked the fundamental question: “Can machines think?”\n\nThe Three Levels of AI:\n\n1. Artificial Narrow Intelligence (ANI) [...] We are living through one of the most significant technological revolutions in human history. Artificial Intelligence has evolved from a niche academic fie

In [20]:
result = executor.invoke({
    "input": "오늘 AI 관련 뉴스 최신으로 5개 요약해주고, 그 내용을 csv로 저장해줘",
    "YYYYMMDD": get_today()
})



> Entering new AgentExecutor chain...

Invoking: `tavily_search` with `{'query': 'AI 최신 뉴스', 'search_depth': 'advanced', 'topic': 'news', 'time_range': 'day'}`


{'query': 'AI 최신 뉴스', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.startupecosystem.ca/news/openai-to-introduce-new-customization-and-content-options-for-chatgpt/', 'title': 'OpenAI to Introduce New Customization and Content Options for ChatGPT - Startup Ecosystem Canada', 'score': 0.5663309, 'published_date': 'Wed, 15 Oct 2025 02:58:12 GMT', 'content': '### Related Posts\n\n7 months ago Global Yahoo’s Strategic AI Moves Under Jim Lanzone\n\n### Yahoo’s Strategic AI Moves Under Jim Lanzone\n\nStartup Eco System21 March 2025\n\n3 months ago Global Paralign Health Launches to Transform Preventative Care\n\n### Paralign Health Launches to Transform Preventative Care\n\nStartup Eco System23 July 2025\n\n6 months ago Global Apple Modifies App Store Rules Amid Epic Games Legal Battle\

In [21]:
result = executor.invoke({
    "input": "오늘 에이전트 AI 관련 뉴스 최신으로 5개 요약해주고, 그 내용을 csv로 저장해줘",
    "YYYYMMDD": get_today()
})



> Entering new AgentExecutor chain...

Invoking: `tavily_search` with `{'query': '에이전트 AI 최신 뉴스', 'search_depth': 'advanced', 'topic': 'news', 'time_range': 'day'}`


{'query': '에이전트 AI 최신 뉴스', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.consultancy.eu/news/amp/12620/how-agentic-ai-can-take-pricing-to-the-next-level', 'title': 'How agentic AI can take pricing to the next level - Consultancy.eu', 'score': 0.62248284, 'published_date': 'Wed, 15 Oct 2025 04:05:35 GMT', 'content': '# How agentic AI can take pricing to the next level\n\n15 October 2025 Consultancy.eu 5 min. read\n\nIn a world where pricing decisions are becoming increasingly complex and dynamic, businesses need intelligent data – and this is where agentic AI comes into play. In terms of opportunities and impact, agentic AI can go beyond traditional AI or even generative AI, write Danilo Zatta and Anders Worsøe Gantzhorn from Valcon. [...] But if the technology is implemented

In [25]:
result = executor.invoke({
    "input": "오늘 Langraph 관련 뉴스 최신으로 5개 요약해주고, 그 내용 5개 모두를 csv로 저장해줘",
    "YYYYMMDD": get_today()
})



> Entering new AgentExecutor chain...

Invoking: `tavily_search` with `{'query': 'Langraph', 'time_range': 'day', 'topic': 'general', 'search_depth': 'advanced'}`


{'query': 'Langraph', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.pgcareers.com/us/en/job/R000139750/R-D-PhD-Summer-Intern-AI-Research-Engineer', 'title': 'R&D PhD Summer Intern - AI Research Engineer - P&G Careers', 'content': 'Preferred Qualifications: Langchain or Langraph application experience. Proficiency in FastAPI. Familiarity with docker container environments. Compensation for', 'score': 0.30442458, 'raw_content': None}, {'url': 'https://www.synechron.com/en-us/careers/jobs/all', 'title': 'Current Openings 115 - Careers Listings | Synechron', 'content': 'Other Technologies\n\n#### Perl Developer\n\nGurugram, India\n\nJR1029729\n\nOther Technologies\n\n#### Platform Engineer – Elastic Search\n\nNew York,NY, United States of America\n\nJR1031539\n\nOther Technologies

## 3. 데이터 베이스 저장 툴

In [6]:
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_community.utilities.sql_database import SQLDatabase

# 1. 모델 설정
llm = ChatOpenAI(
    model = "gpt-4.1-mini",
    temperature = 0
)

db = SQLDatabase.from_uri("sqlite:///../10_agent_result/wanted2.db")
toolkit = SQLDatabaseToolkit(db=db, llm=llm).get_tools()

In [7]:
toolkit

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x0000020EA0044490>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x0000020EA0044490>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x0000020EA0044490>),
 QuerySQLCheckerTool(description='Use this tool to 

In [ ]:
DB_PROMPT = """
너는 '웹 검색 툴 (Tavily)'과 'SQL 데이터베이스 툴 (SQLDatabaseToolkit)'만을 사용하여 정보를 검색하기도 하고,
검색된 데이터를 데이터베이스에 저장하기도 하는 전문 어시스턴트야.

[규칙]
1. 데이터 베
1. 최신 뉴스/링크를 요청받으면 **반드시** '웹 검색 툴 (Tavily)'을 사용하여 관련 정보를 검색한다.
2. 웹 검색으로 데이터를 불러온 후, 이 데이터를 'SQL 데이터베이스 툴'을 사용하여 데이터베이스의 특정 테이블에 저장한다.
3. 만약 테이블이 없다면 테이블을 생성해야한다.
3. 데이터베이스에 저장할 때 사용하는 필드(컬럼)는 다음과 같이 한다:
    - **search_date** (검색 요청 날짜 및 시간)
    - **query** (사용자가 검색 요청한 키워드)
    - **title** (뉴스/링크의 제목)
    - **url** (뉴스/링크의 URL)
    - **source** (정보의 출처 도메인)
    - **snippet** (뉴스/링크의 요약 내용)
4. 데이터베이스 저장 작업이 완료되면, 사용자에게 **저장된 내용 중 가장 최근에 저장된 5개의 항목**하을 테이블 형태로 출력여 저장 성공을 확인시켜준다.
5. 데이터베이스 툴이나 웹 검색 툴을 사용한 모든 과정은 사용자에게 보이지 않게 간결하게 처리하며, **모든 답변은 한국어로 간결하게** 한다.
"""

In [9]:
# 1. 모델 설정
llm = ChatOpenAI(
    model = "gpt-4.1-mini",
    temperature = 0
)

# 2. 도구 생성
tools = toolkit + [tavily_search]

# 3. 프롬프트 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", DB_PROMPT),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

# 4. 단일 에이전트 생성
agent = create_openai_tools_agent(
    llm=llm,
    tools = tools,
    prompt = prompt
)

# 5. excutor 설정
executor = AgentExecutor(
    agent = agent,
    tools = tools,
    verbose = True
)

In [10]:
result = executor.invoke({"input": "최신 NPU 관련 뉴스 5개 찾아서 알려줘"
})



> Entering new AgentExecutor chain...

Invoking: `tavily_search` with `{'query': '최신 NPU 뉴스', 'search_depth': 'advanced', 'time_range': 'month', 'topic': 'news'}`


{'query': '최신 NPU 뉴스', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.cnet.com/tech/mobile/mediateks-next-chip-will-boost-low-power-ai-in-next-years-top-android-phones/', 'title': "MediaTek's Next Chip Will Boost Low-Power AI in Next Year's Top Android Phones - CNET", 'score': 0.4504325, 'published_date': 'Mon, 22 Sep 2025 07:00:00 GMT', 'content': "Given the aggressive adoption of AI on mobile, it's unsurprising that MediaTek also boosted artificial intelligence on its new chip. The Dimensity 9500's NPU 990 has a smattering of upgrades over its predecessor, with 100% faster processing for the smaller 3 billion parameter models often used on smartphones (which typically top out at 7 billion parameters). The NPU (neural processing unit) is also 42% more power efficient when runn

In [ ]:
result = executor.invoke({"invoke": "최신 NPU 관련 뉴스 5개 찾아서 DB에 저장해줘"
})